# 1. Purpose and scope

SCRUM-13 defines the deterministic temporal-validation contract for Favorita. It locks the eight expanding-window fold boundaries, the training-label cutoff, leakage protections, and the final untouched holdout.

This notebook does not train or score a model, calculate forecasting metrics, tune parameters, build the full feature dataset, or run a multi-fold backtest.

## 2. Relationship to SCRUM-11 and SCRUM-12

SCRUM-11 established the leakage-safe feature policy: origin-bounded history, conditional as-of-origin promotion and holiday availability, direct horizons, and sparse observed-row semantics. SCRUM-12 implemented the reusable model-ready feature contract and exact horizons 1 through 16. SCRUM-13 consumes those contracts without changing historical 1-, 7-, 14-, or 28-day feature definitions.

## 3. Approved 16-day forecast contract

- Target: `unit_sales`.
- Forecast origin: end of calendar day `t`.
- Design: direct horizon-aware global forecasting.
- Horizons: exact integer set 1 through 16.
- Forecast dates: `t+1` through `t+16`.
- Recursive prediction feedback: forbidden.
- Grain: `(forecast_origin, forecast_date, store_nbr, item_nbr)`.
- Row semantics: preserve observed source-derived rows; never densify or infer zero demand from absence.

In [1]:
from datetime import timedelta

from pipelines.evaluation.favorita_temporal_validation import (
    APPROVED_FOLDS,
    FINAL_HOLDOUT,
    FORECAST_HORIZONS,
    derive_target_window,
    is_training_target_eligible,
    require_training_target_eligible,
    validate_approved_contract,
    validate_forecast_date_horizon,
)

print({
    "target": "unit_sales",
    "origin": "end of calendar day t",
    "horizons": FORECAST_HORIZONS,
    "design": "direct horizon-aware; no recursive feedback",
})

{'target': 'unit_sales', 'origin': 'end of calendar day t', 'horizons': (1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16), 'design': 'direct horizon-aware; no recursive feedback'}


## 4. Why random splitting is forbidden

Random-row splitting mixes simulated future and past observations, allows later labels or fitted state to influence earlier origins, and breaks the operational end-of-day cutoff. Validation must advance only in chronological order.

## 5. Expanding-window validation method

For fold origin `O`, eligible training history expands from the earliest usable history through labels satisfying `forecast_date <= O`. The validation targets are only `O+1` through `O+16`. An earlier completed validation period may enter a later fold's training history only after those dates are in the later fold's simulated past.

## 6. Approved eight fold origins

The fold-design audit locks eight strictly increasing origins. The reusable module stores only these canonical origin dates and derives each validation boundary from the 16-horizon contract.

In [2]:
for fold in APPROVED_FOLDS:
    print(
        f"Fold {fold.fold_id}: origin={fold.forecast_origin}; "
        f"validation={fold.validation_start} through {fold.validation_end}"
    )

Fold 1: origin=2015-08-31; validation=2015-09-01 through 2015-09-16
Fold 2: origin=2015-12-08; validation=2015-12-09 through 2015-12-24
Fold 3: origin=2016-04-15; validation=2016-04-16 through 2016-05-01
Fold 4: origin=2016-06-30; validation=2016-07-01 through 2016-07-16
Fold 5: origin=2016-08-31; validation=2016-09-01 through 2016-09-16
Fold 6: origin=2016-12-08; validation=2016-12-09 through 2016-12-24
Fold 7: origin=2017-04-15; validation=2017-04-16 through 2017-05-01
Fold 8: origin=2017-06-30; validation=2017-07-01 through 2017-07-16


## 7. Exact 16-day validation windows

Every validation window begins one calendar day after its origin, ends 16 calendar days after its origin, contains exactly 16 inclusive dates, and is disjoint from every other approved window.

In [3]:
for fold in APPROVED_FOLDS:
    expected_start, expected_end = derive_target_window(fold.forecast_origin)
    assert fold.validation_start == expected_start
    assert fold.validation_end == expected_end
    assert (fold.validation_end - fold.validation_start).days + 1 == 16
assert all(
    current.validation_end < following.validation_start
    for current, following in zip(APPROVED_FOLDS, APPROVED_FOLDS[1:])
)
print("8 folds validated: each window has 16 dates and windows do not overlap.")

8 folds validated: each window has 16 dates and windows do not overlap.


## 8. Training-label cutoff: `forecast_date <= fold_origin`

A training example is eligible only when its target label is already known at the simulated fold origin. Testing only `training_example.forecast_origin < O` is insufficient because that row's `forecast_date` could still be later than `O`.

In [4]:
example_origin = APPROVED_FOLDS[0].forecast_origin
cutoff_checks = {
    "before origin": is_training_target_eligible(
        example_origin - timedelta(days=1), example_origin
    ),
    "on origin": is_training_target_eligible(example_origin, example_origin),
    "after origin": is_training_target_eligible(
        example_origin + timedelta(days=1), example_origin
    ),
}
assert cutoff_checks == {
    "before origin": True, "on origin": True, "after origin": False
}
try:
    require_training_target_eligible(
        example_origin + timedelta(days=1), example_origin
    )
except ValueError as error:
    print(cutoff_checks)
    print(f"Post-origin target rejected: {error}")
else:
    raise AssertionError("A post-origin training target was not rejected")

{'before origin': True, 'on origin': True, 'after origin': False}
Post-origin target rejected: Training forecast_date must be on or before the fold origin


## 9. Leakage controls

The contract forbids: random-row splitting; post-origin training labels; fold overlap; holdout overlap; future actual `unit_sales`, transactions, or oil prices; features constructed at a later origin; learned preprocessing or category-vocabulary fitting on validation rows; duplicate forecast-example grain; recursive predictions; and densification or missing-row-to-zero inference.

Promotion and holiday target-date fields retain their existing SCRUM-11/SCRUM-12 as-of-origin availability rules. This notebook does not broaden those assumptions.

## 10. Final untouched holdout

The protected holdout origin is `2017-07-30`; target dates are `2017-07-31` through `2017-08-15`. The holdout is excluded from validation folds, design tuning, model selection, preprocessing fitting, metric or threshold selection, and hyperparameter tuning.

In [5]:
assert FINAL_HOLDOUT.forecast_origin.isoformat() == "2017-07-30"
assert FINAL_HOLDOUT.holdout_start.isoformat() == "2017-07-31"
assert FINAL_HOLDOUT.holdout_end.isoformat() == "2017-08-15"
assert all(
    fold.validation_end < FINAL_HOLDOUT.holdout_start
    for fold in APPROVED_FOLDS
)
print(FINAL_HOLDOUT)

HoldoutWindow(forecast_origin=datetime.date(2017, 7, 30), holdout_start=datetime.date(2017, 7, 31), holdout_end=datetime.date(2017, 8, 15))


## 11. Fold-contract executable validation

The module validates exact fold count, ordered identifiers, strict chronology, date equations, inclusive durations, fold non-overlap, holdout protection, exact horizons, and direct forecast-date equations. It raises `ValueError` when a boundary is violated.

In [6]:
validate_approved_contract()
for fold in APPROVED_FOLDS:
    for horizon in FORECAST_HORIZONS:
        validate_forecast_date_horizon(
            fold.forecast_origin,
            fold.forecast_origin + timedelta(days=horizon),
            horizon,
        )
print(
    f"Contract valid: {len(APPROVED_FOLDS)} folds, "
    f"{len(FORECAST_HORIZONS)} direct horizons, protected holdout."
)

Contract valid: 8 folds, 16 direct horizons, protected holdout.


## 12. Explicit boundaries for future work

- SCRUM-14 owns forecasting baselines.
- SCRUM-15 owns the first global forecasting model.
- SCRUM-16 owns actual expanding-window backtesting orchestration.
- SCRUM-17 owns forecasting metrics and aggregation policy.

No work from those tickets is implemented here.

## 13. SCRUM-13 completion checklist

The checklist below reports executable contract coverage only. It makes no claim about model quality, metric evaluation, completed backtesting, production readiness, research completion, or cloud deployment readiness.

In [7]:
completion_checks = {
    "exact eight approved folds": len(APPROVED_FOLDS) == 8,
    "exact horizons 1 through 16": FORECAST_HORIZONS == tuple(range(1, 17)),
    "training cutoff enforced": not is_training_target_eligible(
        APPROVED_FOLDS[0].forecast_origin + timedelta(days=1),
        APPROVED_FOLDS[0].forecast_origin,
    ),
    "holdout protected": all(
        fold.validation_end < FINAL_HOLDOUT.holdout_start
        for fold in APPROVED_FOLDS
    ),
    "no model, metric, or backtest execution": True,
}
assert all(completion_checks.values())
for check, passed in completion_checks.items():
    print(f"{check}: {passed}")

exact eight approved folds: True
exact horizons 1 through 16: True
training cutoff enforced: True
holdout protected: True
no model, metric, or backtest execution: True
